In [ ]:
from models import VDPResNet18
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torch
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
from copy import deepcopy
# Train on food, validate on flowers

trans = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

train_ds = datasets.Food101(root="/home/jw7630/repos/sparsity/data", split="train", transform=trans, download=True)
test_ds = datasets.Food101(root="/home/jw7630/repos/sparsity/data", split="test", transform=trans, download=True)
ood_ds = datasets.Flowers102(root="/home/jw7630/repos/sparsity/data", split="train", transform=trans, download=True)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=256, shuffle=False)
ood_loader = torch.utils.data.DataLoader(ood_ds, batch_size=256, shuffle=False)


model_vdp = VDPResNet18(num_classes=1000)
model_vdp(next(iter(train_loader))[0])
device = "cpu"

In [15]:
# Load the ResNet18 model

from torchvision.models import resnet18
model = resnet18(pretrained=True)
model_vdp.load_weights(model)
model_vdp.assign_bn_stats(model)

/home/jw7630/repos/vdp_modelzoo/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/jw7630/repos/vdp_modelzoo/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


✅ Copied running stats for 20 batch-norm layers.


In [23]:
from layers import VDPIntermediateLinear
model_vdp.fc = VDPIntermediateLinear(101, bias=True)
initialize = model_vdp(torch.ones(1,3,224,224))

# Scenarios:
# 1) Train only fc mu with CE
# 2) Train fc mu and sigma with VDP losses (3)
# 3) Train resnet sigmas and fc mu and sigma with VDP losses (3)


In [ ]:
lr = 0.0001
kl_factor = 0.000001
EPOCHS = 50
criterion = nn.CrossEntropyLoss()

model = deepcopy(model_vdp)
# Scenraio 1
optimizer1 = optim.Adam(model_vdp.fc.parameters(), lr=lr)
for epoch in range(EPOCHS):
    # TRAINING
    model.train()
    train_loss, train_accuracy = 0.0, 0.0
    for i, (images, labels) in enumerate(train_loader):
        batch_size = images.size(0)
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        y_hat = model(images)
        loss = criterion(y_hat, labels)
        optimizer1.zero_grad()
        loss.backward()
        optimizer1.step()
        train_loss += loss.item() * batch_size
        train_accuracy += (y_hat.argmax(dim=1) == labels).float().sum().item()
    train_loss /= len(train_loader.dataset)
    train_accuracy /= len(train_loader.dataset)
   
   # EVALUATION
    test_loss, test_accuracy = 0.0, 0.0
    model.eval()
    for test_images, test_labels in test_loader:
        batch_size = test_images.size(0)
        test_images = test_images.to(device)
        test_labels = test_labels.to(device)
        test_outputs = model(test_images)

        test_loss += criterion(test_outputs, test_labels).detach().item() * batch_size
        test_accuracy += (test_outputs.argmax(dim=1) == test_labels).float().sum().cpu().item()

        del test_images, test_labels, test_outputs
        torch.cuda.empty_cache()
        optimizer1.zero_grad()

    test_loss /= len(test_loader.dataset)
    test_accuracy /= len(test_loader.dataset)
    


KeyboardInterrupt: 

In [10]:
model_vdp.eval()
model.eval()
input_tensor = torch.randn(1, 3, 224, 224)
mu, sigma, kl, mu11, mu12, mu21, mu22, mu31, mu32, mu41, mu42 = model_vdp(input_tensor)
a = model(input_tensor)

In [14]:
(torch.abs(mu-a)<0.00001).all()

tensor(True)

In [15]:
mu41.sum()

tensor(2757.6340, grad_fn=<SumBackward0>)

In [8]:
mu.sum()

tensor(0.0462, grad_fn=<SumBackward0>)

In [ ]:
(resnet18().children()[-1:])

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [4]:
a - mu

tensor([[-2.6355e-03,  3.0009e-03,  6.5476e-04, -2.6908e-02,  6.3636e-03,
          1.3260e-02, -1.1179e-02,  2.0642e-02, -3.6362e-03, -1.2324e-02,
         -1.2631e-02, -7.2041e-03, -1.9322e-02, -2.4961e-02, -1.1887e-02,
         -8.3280e-03, -9.5758e-03, -1.6658e-02,  9.1796e-03, -1.5358e-02,
          7.1325e-03,  3.0736e-02,  1.3241e-02, -7.7503e-03,  4.7398e-03,
          1.1177e-02,  1.5951e-02, -1.6710e-02, -1.0142e-03, -3.7153e-03,
          6.5274e-03, -1.2039e-02,  9.0458e-03, -8.3351e-04,  8.8625e-03,
         -2.6304e-02, -1.4585e-02,  2.9445e-03,  2.9727e-03, -1.9125e-02,
         -4.7915e-03,  1.3828e-02,  9.8794e-03, -1.8421e-02,  1.9734e-02,
          1.6968e-03,  1.2423e-02, -5.5866e-03, -1.0614e-02,  3.9673e-04,
          4.2729e-03, -1.3301e-02,  2.0662e-02,  1.6963e-02,  2.7962e-03,
          7.4100e-04,  1.3167e-02,  3.2177e-03,  1.0458e-02,  1.6508e-02,
          9.1696e-04,  3.9387e-03, -5.6553e-03,  1.9372e-02,  7.5219e-03,
          1.3436e-02, -1.3185e-02, -1.

In [25]:
mu12.sum()

tensor(-9815.4414, grad_fn=<SumBackward0>)

In [17]:
list(model.modules())

[ResNet(
   (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
   (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (relu): ReLU(inplace=True)
   (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
   (layer1): Sequential(
     (0): BasicBlock(
       (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
       (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
       (relu): ReLU(inplace=True)
       (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
       (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     )
     (1): BasicBlock(
       (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
       (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
       (relu): ReLU

In [18]:
import torch.nn as nn
from typing import List, Tuple

def flatten(model: nn.Module) -> List[nn.Module]:
    """Return all submodules except the model itself, in pre‐order."""
    return list(model.modules())[1:]

def diff_module_lists(a: List[nn.Module], b: List[nn.Module]) -> List[Tuple[int, str, str]]:
    """
    Compare two lists of modules by their class names.
    Returns a list of (index, a_name, b_name) for every position where they differ.
    """
    diffs = []
    n = max(len(a), len(b))
    for i in range(n):
        name_a = type(a[i]).__name__ if i < len(a) else "<MISSING>"
        name_b = type(b[i]).__name__ if i < len(b) else "<MISSING>"
        if name_a != name_b:
            diffs.append((i, name_a, name_b))
    return diffs

# --- usage example ---

import torchvision.models as models

# 1) flatten both
custom_flat = flatten(model_vdp)      # your VDPResNet18 instance
res18_flat  = flatten(models.resnet18())  # pretrained ResNet-18

# 2) diff
mismatches = diff_module_lists(custom_flat, res18_flat)

# 3) report
if not mismatches:
    print("✅ All module types line up perfectly.")
else:
    print(f"❌ Found {len(mismatches)} mismatches:")
    for idx, a_name, b_name in mismatches:
        print(f"  Index {idx:3d}: custom is {a_name:<20s} | resnet18 is {b_name}")


❌ Found 67 mismatches:
  Index   0: custom is VDPFirstConv         | resnet18 is Conv2d
  Index   1: custom is VDP_BatchNorm2D      | resnet18 is BatchNorm2d
  Index   2: custom is VDP_ReLU             | resnet18 is ReLU
  Index   3: custom is VDPMaxPooling        | resnet18 is MaxPool2d
  Index   4: custom is VDPResnetBasicBlock  | resnet18 is Sequential
  Index   5: custom is VDPIntermediateConv  | resnet18 is BasicBlock
  Index   6: custom is VDP_BatchNorm2D      | resnet18 is Conv2d
  Index   7: custom is VDP_ReLU             | resnet18 is BatchNorm2d
  Index   8: custom is VDPIntermediateConv  | resnet18 is ReLU
  Index   9: custom is VDP_BatchNorm2D      | resnet18 is Conv2d
  Index  10: custom is VDPResnetBasicBlock  | resnet18 is BatchNorm2d
  Index  11: custom is VDPIntermediateConv  | resnet18 is BasicBlock
  Index  12: custom is VDP_BatchNorm2D      | resnet18 is Conv2d
  Index  13: custom is VDP_ReLU             | resnet18 is BatchNorm2d
  Index  14: custom is VDPIntermedia

In [21]:
import torch.nn as nn
from typing import List, Tuple

def flatten_no_sequential(model: nn.Module) -> List[nn.Module]:
    """
    Return all submodules except the model itself and skip nn.Sequential containers,
    in pre‐order.
    """
    # all submodules, drop the root
    mods = list(model.modules())[1:]
    # filter out Sequentials
    return [m for m in mods if not isinstance(m, nn.Sequential)]

def diff_module_lists(a: List[nn.Module], b: List[nn.Module]) -> List[Tuple[int, str, str]]:
    diffs = []
    n = max(len(a), len(b))
    for i in range(n):
        name_a = type(a[i]).__name__ if i < len(a) else "<MISSING>"
        name_b = type(b[i]).__name__ if i < len(b) else "<MISSING>"
        if name_a != name_b:
            diffs.append((i, name_a, name_b))
    return diffs

# --- usage example ---

import torchvision.models as models

custom_flat = flatten_no_sequential(model_vdp)
res18_flat  = flatten_no_sequential(models.resnet18())

mismatches = diff_module_lists(custom_flat, res18_flat)

if not mismatches:
    print("✅ All module types line up perfectly.")
else:
    print(f"❌ Found {len(mismatches)} mismatches:")
    for idx, a_name, b_name in mismatches:
        print(f"  Index {idx:3d}: custom is {a_name:<20s} | resnet18 is {b_name}")


❌ Found 60 mismatches:
  Index   0: custom is VDPFirstConv         | resnet18 is Conv2d
  Index   1: custom is VDP_BatchNorm2D      | resnet18 is BatchNorm2d
  Index   2: custom is VDP_ReLU             | resnet18 is ReLU
  Index   3: custom is VDPMaxPooling        | resnet18 is MaxPool2d
  Index   4: custom is VDPResnetBasicBlock  | resnet18 is BasicBlock
  Index   5: custom is VDPIntermediateConv  | resnet18 is Conv2d
  Index   6: custom is VDP_BatchNorm2D      | resnet18 is BatchNorm2d
  Index   7: custom is VDP_ReLU             | resnet18 is ReLU
  Index   8: custom is VDPIntermediateConv  | resnet18 is Conv2d
  Index   9: custom is VDP_BatchNorm2D      | resnet18 is BatchNorm2d
  Index  10: custom is VDPResnetBasicBlock  | resnet18 is BasicBlock
  Index  11: custom is VDPIntermediateConv  | resnet18 is Conv2d
  Index  12: custom is VDP_BatchNorm2D      | resnet18 is BatchNorm2d
  Index  13: custom is VDP_ReLU             | resnet18 is ReLU
  Index  14: custom is VDPIntermediateConv

In [10]:
nn.Sequential(*list(model_vdp.children())[:2])(input_tensor)[0][0][0][0]

TypeError: VDP_BatchNorm2D.forward() missing 1 required positional argument: 'sigma_in'

In [5]:
list(model_vdp.named_children())

[('conv1', VDPFirstConv()),
 ('bn1', VDP_BatchNorm2D()),
 ('relu', VDP_ReLU()),
 ('maxpool', VDPMaxPooling()),
 ('layer11',
  VDPResnetBasicBlock(
    (conv1): VDPIntermediateConv()
    (bn1): VDP_BatchNorm2D()
    (relu): VDP_ReLU()
    (conv2): VDPIntermediateConv()
    (bn2): VDP_BatchNorm2D()
  )),
 ('layer12',
  VDPResnetBasicBlock(
    (conv1): VDPIntermediateConv()
    (bn1): VDP_BatchNorm2D()
    (relu): VDP_ReLU()
    (conv2): VDPIntermediateConv()
    (bn2): VDP_BatchNorm2D()
  )),
 ('layer21',
  VDPResnetBasicBlock(
    (conv1): VDPIntermediateConv()
    (bn1): VDP_BatchNorm2D()
    (relu): VDP_ReLU()
    (conv2): VDPIntermediateConv()
    (bn2): VDP_BatchNorm2D()
    (shortcut_conv): VDPIntermediateConv()
    (shortcut_batchnorm): VDP_BatchNorm2D()
  )),
 ('layer22',
  VDPResnetBasicBlock(
    (conv1): VDPIntermediateConv()
    (bn1): VDP_BatchNorm2D()
    (relu): VDP_ReLU()
    (conv2): VDPIntermediateConv()
    (bn2): VDP_BatchNorm2D()
  )),
 ('layer31',
  VDPResnetBasicB

In [72]:
list(model.named_children())

[('conv1',
  Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)),
 ('bn1',
  BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)),
 ('relu', ReLU(inplace=True)),
 ('maxpool',
  MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)),
 ('layer1',
  Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=

In [ ]:

torch.ones(1, 3, 224, 224) 

tensor([[[[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]],

         [[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]],

         [[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]]]])

In [20]:
list(model.named_children())[4]

('layer1',
 Sequential(
   (0): BasicBlock(
     (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
     (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (relu): ReLU(inplace=True)
     (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
     (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   )
   (1): BasicBlock(
     (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
     (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (relu): ReLU(inplace=True)
     (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
     (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   )
 ))

In [33]:
input_tensor

tensor([[[[ 1.7879e+00, -1.4814e-01,  9.3156e-01,  ..., -8.6137e-01,
            5.7643e-01,  8.7667e-01],
          [-6.6453e-01,  8.6051e-01, -7.9904e-01,  ..., -2.1511e-01,
            1.4049e-01,  5.2125e-01],
          [ 3.0891e-01, -2.9101e-01,  9.1755e-01,  ...,  1.3256e-01,
            9.1038e-01, -5.2769e-01],
          ...,
          [-5.6533e-01,  1.6618e-01, -2.4554e-01,  ..., -9.2067e-01,
           -1.6707e+00,  1.4045e+00],
          [-1.0560e-01, -1.0097e-01, -2.2627e-02,  ...,  9.9573e-01,
            1.6249e+00,  2.4916e-01],
          [ 5.4789e-01,  2.1911e-01,  2.4297e+00,  ...,  2.6590e-01,
            1.7211e+00,  7.8554e-01]],

         [[-8.6890e-01, -7.8453e-01,  2.9792e-01,  ...,  1.4593e+00,
           -7.6414e-01, -3.2432e-01],
          [ 8.7094e-01, -7.8152e-02,  7.3026e-02,  ..., -8.4006e-01,
            3.1633e-01, -8.9714e-01],
          [ 6.2199e-01, -1.0282e-01,  3.8303e-01,  ..., -3.8630e-01,
           -1.3475e+00,  7.3804e-01],
          ...,
     

In [67]:
nn.Sequential(*list(model.children())[:2])(input_tensor)[0][0][0]

tensor([-0.3304, -0.5025, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989,
        -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989,
        -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989,
        -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989,
        -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989,
        -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989,
        -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989,
        -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989,
        -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989,
        -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989,
        -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989,
        -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989, -0.4989,
        -0.4989, -0.4989, -0.4989, -0.49

In [69]:
for i in range(len(nn.Sequential(*list(model.children())))):
    print(nn.Sequential(*list(model.children())[:i])(input_tensor).shape)
    print(nn.Sequential(*list(model.children())[:i])(input_tensor)[0][0][0])

torch.Size([1, 3, 224, 224])
tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1.,

In [29]:
mu12

tensor([[[[0.4854, 1.1998, 1.2005,  ..., 0.4962, 0.9400, 0.9044],
          [0.8847, 1.1784, 1.1770,  ..., 0.3679, 1.0061, 0.9658],
          [0.8984, 1.1607, 1.1375,  ..., 0.9796, 1.6840, 1.6420],
          ...,
          [0.8701, 1.4253, 1.0691,  ..., 0.4817, 1.3323, 1.1810],
          [0.3232, 1.2158, 1.1825,  ..., 1.7526, 1.0059, 1.1043],
          [0.9565, 1.5203, 1.6018,  ..., 0.9616, 0.8995, 0.4550]],

         [[0.0431, 0.0000, 0.0000,  ..., 0.0503, 0.1347, 0.1572],
          [0.0942, 0.0161, 0.1373,  ..., 0.1295, 0.2378, 0.1993],
          [0.1004, 0.0300, 0.2917,  ..., 0.2896, 0.3602, 0.2097],
          ...,
          [0.0725, 0.0652, 0.2218,  ..., 0.0000, 0.0000, 0.0453],
          [0.1391, 0.0903, 0.1656,  ..., 0.1723, 0.0000, 0.2038],
          [0.2785, 0.1853, 0.2754,  ..., 0.2328, 0.2997, 0.3802]],

         [[0.1115, 0.1975, 0.2122,  ..., 0.2533, 0.2094, 0.1075],
          [0.2432, 0.3720, 0.3708,  ..., 0.3958, 0.3570, 0.2166],
          [0.2925, 0.4145, 0.3804,  ..., 0

In [ ]:
for a,b in zip(model.children(), model_vdp.children()):
    if hasattr(a[1]
    print((a==b).all())

tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)


In [16]:
mu

tensor([[-1.9826e-01, -9.9367e-02, -3.5798e-01, -6.1055e-01, -2.2135e-01,
         -5.2465e-02, -2.3439e-01, -3.9280e-02, -1.2032e-01, -2.0864e-01,
         -3.9080e-01, -2.8258e-01, -1.3051e-01, -2.1134e-01,  1.8396e-02,
         -9.9103e-02, -2.3124e-01, -1.5750e-01,  4.6334e-02, -2.1731e-01,
         -1.2529e-01, -2.6458e-01, -3.8158e-01,  5.4299e-03, -4.0988e-01,
         -3.8302e-01, -3.2048e-01, -4.0429e-01, -3.2123e-01, -1.1056e-01,
         -2.9645e-01, -3.1304e-01, -9.3225e-02, -1.3663e-01,  1.2312e-01,
         -2.4027e-01,  1.8716e-01, -5.0392e-01, -3.8516e-01, -7.4548e-02,
         -1.5290e-01, -1.7242e-01, -2.3448e-01, -3.7874e-01, -1.3340e-01,
         -1.1435e-01, -1.4500e-01, -2.1513e-01, -4.6504e-01, -2.3996e-01,
         -1.2924e-01, -1.0098e-01, -3.3419e-04, -1.1870e-01, -1.1908e-01,
         -3.8744e-01, -1.1329e-03, -4.3404e-01, -3.2418e-02,  5.6944e-02,
          1.5246e-01, -1.4214e-01, -9.2738e-02,  1.9740e-02,  6.9279e-02,
         -2.9443e-02,  4.1323e-02, -1.

In [ ]:
# Grab plain lists of modules (not (name,module) tuples)
vdp_mods   = list(model_vdp.children())
model_mods = list(model.children())

j = 0
for vmod in vdp_mods:
    # only process modules that actually have running_mean/running_var
    if hasattr(vmod, "running_mean"):
        # find the next BN‐like module in model_mods
        while j < len(model_mods) and not hasattr(model_mods[j], "running_mean"):
            j += 1
        if j >= len(model_mods):
            raise RuntimeError("Ran out of BatchNorm layers in the source model!")
        # copy stats
        vmod.running_mean.data.copy_(model_mods[j].running_mean)
        vmod.running_var.data.copy_(model_mods[j].running_var)
        j += 1



RuntimeError: output with shape [1, 64, 1, 1] doesn't match the broadcast shape [1, 64, 1, 64]

In [10]:
list(m2.named_children())[1][1].running_mean

tensor([ 2.7681e-03, -2.5769e-02,  2.1254e-07, -8.4605e-02,  2.1121e-08,
         4.9691e-04, -2.2408e-02, -1.1582e-07, -4.8239e-03,  2.7507e-07,
         3.9582e-02,  3.1994e-02, -3.7490e-02, -1.3716e-06,  6.6002e-03,
         4.3782e-03,  6.4797e-02,  1.1176e-01,  3.6002e-02, -7.5075e-02,
        -3.8240e-02,  8.4358e-02, -5.2287e-02, -1.1799e-02,  1.3019e-03,
         3.2172e-02, -1.7784e-02, -9.1009e-02,  1.1319e-01, -4.1632e-02,
         8.7302e-03,  2.9693e-02, -7.0502e-02, -3.4847e-03,  1.0977e-01,
        -1.7341e-03, -5.9423e-08,  2.9330e-02, -7.8553e-09,  6.7320e-03,
        -3.7100e-03,  1.6028e-02, -2.7883e-02,  2.6593e-02,  2.8475e-02,
        -1.2735e-01,  4.4617e-02,  2.6329e-02,  2.1454e-08, -1.7045e-02,
        -3.5617e-03, -4.5841e-02,  6.3876e-02,  1.5220e-02, -3.8511e-02,
        -1.6428e-02, -1.6569e-02,  5.6057e-02, -8.0306e-02, -2.6646e-03,
        -4.1718e-02,  1.2611e-01, -4.9237e-02, -1.3261e-02])

In [9]:
list(model_vdp.named_children())[1][1].running_mean

tensor([ 2.7681e-03, -2.5769e-02,  2.1254e-07, -8.4605e-02,  2.1121e-08,
         4.9691e-04, -2.2408e-02, -1.1582e-07, -4.8239e-03,  2.7507e-07,
         3.9582e-02,  3.1994e-02, -3.7490e-02, -1.3716e-06,  6.6002e-03,
         4.3782e-03,  6.4797e-02,  1.1176e-01,  3.6002e-02, -7.5075e-02,
        -3.8240e-02,  8.4358e-02, -5.2287e-02, -1.1799e-02,  1.3019e-03,
         3.2172e-02, -1.7784e-02, -9.1009e-02,  1.1319e-01, -4.1632e-02,
         8.7302e-03,  2.9693e-02, -7.0502e-02, -3.4847e-03,  1.0977e-01,
        -1.7341e-03, -5.9423e-08,  2.9330e-02, -7.8553e-09,  6.7320e-03,
        -3.7100e-03,  1.6028e-02, -2.7883e-02,  2.6593e-02,  2.8475e-02,
        -1.2735e-01,  4.4617e-02,  2.6329e-02,  2.1454e-08, -1.7045e-02,
        -3.5617e-03, -4.5841e-02,  6.3876e-02,  1.5220e-02, -3.8511e-02,
        -1.6428e-02, -1.6569e-02,  5.6057e-02, -8.0306e-02, -2.6646e-03,
        -4.1718e-02,  1.2611e-01, -4.9237e-02, -1.3261e-02],
       grad_fn=<AddBackward0>)

In [51]:
list(m2.named_children())[1][1].running_mean

tensor([ 2.7681e-03, -2.5769e-02,  2.1254e-07, -8.4605e-02,  2.1121e-08,
         4.9691e-04, -2.2408e-02, -1.1582e-07, -4.8239e-03,  2.7507e-07,
         3.9582e-02,  3.1994e-02, -3.7490e-02, -1.3716e-06,  6.6002e-03,
         4.3782e-03,  6.4797e-02,  1.1176e-01,  3.6002e-02, -7.5075e-02,
        -3.8240e-02,  8.4358e-02, -5.2287e-02, -1.1799e-02,  1.3019e-03,
         3.2172e-02, -1.7784e-02, -9.1009e-02,  1.1319e-01, -4.1632e-02,
         8.7302e-03,  2.9693e-02, -7.0502e-02, -3.4847e-03,  1.0977e-01,
        -1.7341e-03, -5.9423e-08,  2.9330e-02, -7.8553e-09,  6.7320e-03,
        -3.7100e-03,  1.6028e-02, -2.7883e-02,  2.6593e-02,  2.8475e-02,
        -1.2735e-01,  4.4617e-02,  2.6329e-02,  2.1454e-08, -1.7045e-02,
        -3.5617e-03, -4.5841e-02,  6.3876e-02,  1.5220e-02, -3.8511e-02,
        -1.6428e-02, -1.6569e-02,  5.6057e-02, -8.0306e-02, -2.6646e-03,
        -4.1718e-02,  1.2611e-01, -4.9237e-02, -1.3261e-02])

In [ ]:
mu, sigma, kl, mu11, mu12, mu21, mu22, mu31, mu32, mu41, mu42 = model_vdp(input_tensor)
a = model(input_tensor)

In [ ]:
a = model(input_tensor)

In [ ]:
mu == a

tensor([[ 5.4836e-01,  2.9538e+00,  2.7406e+00,  2.9007e+00,  4.7830e+00,
          4.3249e+00,  4.3949e+00,  6.6636e-01, -6.0385e-01, -3.7139e-01,
         -2.5439e-01,  1.3105e+00,  6.9156e-01,  1.6218e+00,  1.9237e+00,
          1.1077e+00, -4.6344e-01,  3.1975e-03,  1.7705e+00,  6.6764e-01,
          3.0968e-01,  3.1757e-01,  1.7395e+00,  9.2310e-01,  2.5345e-02,
          2.8925e-01,  9.9213e-01,  7.3524e-01,  4.3965e-01,  8.8451e-01,
          1.4456e+00,  3.0022e-01, -5.1305e-01,  2.9465e+00,  3.5068e+00,
          1.0808e+00,  6.8555e-01,  5.5450e-01,  3.5402e-01,  1.9197e+00,
          1.4669e+00,  1.0283e+00,  1.3777e+00,  9.2580e-01,  1.6333e+00,
          3.6065e-01,  1.7832e+00, -1.0988e+00,  1.6086e+00,  1.0120e+00,
          2.2018e+00, -1.6215e+00,  9.8114e-01,  1.1475e+00, -1.3619e-01,
          1.3617e+00, -7.3222e-01,  6.1018e-01,  2.3578e+00,  5.0888e-01,
          1.5949e+00, -3.8855e-01, -9.7181e-02, -8.7259e-01,  4.2713e-01,
          4.6247e+00,  1.9143e-01, -6.

In [24]:
list(model.named_children())[1][1].running_mean

tensor([ 2.7681e-03, -2.5769e-02,  2.1254e-07, -8.4605e-02,  2.1121e-08,
         4.9691e-04, -2.2408e-02, -1.1582e-07, -4.8239e-03,  2.7507e-07,
         3.9582e-02,  3.1994e-02, -3.7490e-02, -1.3716e-06,  6.6002e-03,
         4.3782e-03,  6.4797e-02,  1.1176e-01,  3.6002e-02, -7.5075e-02,
        -3.8240e-02,  8.4358e-02, -5.2287e-02, -1.1799e-02,  1.3019e-03,
         3.2172e-02, -1.7784e-02, -9.1009e-02,  1.1319e-01, -4.1632e-02,
         8.7302e-03,  2.9693e-02, -7.0502e-02, -3.4847e-03,  1.0977e-01,
        -1.7341e-03, -5.9423e-08,  2.9330e-02, -7.8553e-09,  6.7320e-03,
        -3.7100e-03,  1.6028e-02, -2.7883e-02,  2.6593e-02,  2.8475e-02,
        -1.2735e-01,  4.4617e-02,  2.6329e-02,  2.1454e-08, -1.7045e-02,
        -3.5617e-03, -4.5841e-02,  6.3876e-02,  1.5220e-02, -3.8511e-02,
        -1.6428e-02, -1.6569e-02,  5.6057e-02, -8.0306e-02, -2.6646e-03,
        -4.1718e-02,  1.2611e-01, -4.9237e-02, -1.3261e-02])

In [23]:
list(model_vdp.named_children())[1][1].running_mean.squeeze()

tensor([ 0.0203, -0.0214, -0.0382,  0.0048, -0.0063,  0.0099,  0.0104, -0.0040,
        -0.0401,  0.0071, -0.0081,  0.0059, -0.0078,  0.0186, -0.0155, -0.0266,
         0.0188, -0.0068,  0.0181,  0.0023,  0.0046, -0.0032,  0.0119,  0.0169,
         0.0201,  0.0164, -0.0229, -0.0028, -0.0153, -0.0204, -0.0009,  0.0246,
         0.0014, -0.0068, -0.0106,  0.0090, -0.0216,  0.0151,  0.0276,  0.0042,
         0.0159,  0.0086,  0.0327,  0.0011,  0.0137, -0.0054, -0.0051,  0.0040,
         0.0149,  0.0022,  0.0086,  0.0111, -0.0092,  0.0319,  0.0038,  0.0113,
         0.0100,  0.0023, -0.0228, -0.0012,  0.0014,  0.0043,  0.0168,  0.0086])

In [15]:
a.shape

torch.Size([1, 1000])

In [16]:
mu.shape

torch.Size([1, 1000])

In [24]:




# Dictionary to store the activations of each layer
activations = {}

# Function to register hooks on the specific layers (layer1.0, layer1.1, etc.)
def register_hooks(model):
    def hook_fn(module, input, output):
        layer_name = module.__class__.__name__
        activations[layer_name] = output

    # Register hooks for the specific layers in ResNet18
    model.layer1[0].register_forward_hook(hook_fn)  # layer1.0
    # model.layer1[1].register_forward_hook(hook_fn)  # layer1.1
    # model.layer2[0].register_forward_hook(hook_fn)  # layer2.0
    # model.layer2[1].register_forward_hook(hook_fn)  # layer2.1
    # model.layer3[0].register_forward_hook(hook_fn)  # layer3.0
    # model.layer3[1].register_forward_hook(hook_fn)  # layer3.1
    # model.layer4[0].register_forward_hook(hook_fn)  # layer4.0
    # model.layer4[1].register_forward_hook(hook_fn)  # layer4.1

# Register hooks to capture activations after the specific layers
register_hooks(model)

# Example input tensor (e.g., image with 3 channels, 224x224 size)

# Forward pass
output = model(input_tensor)

# Print the activations (after each specified layer)
for layer_name, activation in activations.items():
    print(f"Layer: {layer_name}, Activation shape: {activation.shape}")


Layer: BasicBlock, Activation shape: torch.Size([1, 64, 56, 56])


In [42]:
mu, sigma, kl = model_vdp.conv1(input_tensor)

In [6]:
model_vdp.bn1.bias

Parameter containing:
tensor([ 2.3072e-01,  2.5382e-01, -1.0543e-06, -6.6439e-01, -1.6571e-08,
         1.6152e-01,  4.5450e-01, -4.3020e-07,  3.0051e-01, -8.0052e-06,
         3.4942e-01,  3.1148e-01, -2.4953e-01, -3.4749e-05,  1.0773e-01,
         2.1897e-01,  3.8141e-01, -5.2988e-01, -6.2864e-01,  5.7140e-01,
         2.9985e-01,  5.8430e-01,  4.8202e-01,  3.2853e-01,  1.9672e-01,
         1.9496e-01,  1.5215e-01,  8.5522e-02,  5.1314e-01,  1.5237e-02,
         1.6644e-01,  3.3239e-01,  2.4921e-01,  4.4337e-01, -2.8017e-01,
        -2.0385e-02, -2.4507e-07,  3.2134e-01, -4.9152e-08,  2.3777e-01,
         2.3291e-01,  3.1527e-01,  4.2776e-01,  2.9313e-01,  2.6379e-01,
         6.7598e-01,  4.2910e-01,  3.4566e-01, -8.6909e-08,  2.4729e-01,
         3.0316e-01,  6.1577e-01,  3.9835e-01,  3.3207e-01, -4.1219e-01,
         3.7807e-01,  1.7895e-01,  2.5748e-01, -4.4908e-01,  2.1306e-01,
         5.6934e-01,  5.7274e-01, -4.0238e-01,  2.3406e-01],
       requires_grad=True)

In [8]:
print("Running mean of layer2.0.bn1:", model_vdp.bn1.running_mean)
print("Running variance of layer2.0.bn1:", model.layer2[0].bn1.running_var)

Running mean of layer2.0.bn1: tensor([[[[ 0.0203]],

         [[-0.0214]],

         [[-0.0382]],

         [[ 0.0048]],

         [[-0.0063]],

         [[ 0.0099]],

         [[ 0.0104]],

         [[-0.0040]],

         [[-0.0401]],

         [[ 0.0071]],

         [[-0.0081]],

         [[ 0.0059]],

         [[-0.0078]],

         [[ 0.0186]],

         [[-0.0155]],

         [[-0.0266]],

         [[ 0.0188]],

         [[-0.0068]],

         [[ 0.0181]],

         [[ 0.0023]],

         [[ 0.0046]],

         [[-0.0032]],

         [[ 0.0119]],

         [[ 0.0169]],

         [[ 0.0201]],

         [[ 0.0164]],

         [[-0.0229]],

         [[-0.0028]],

         [[-0.0153]],

         [[-0.0204]],

         [[-0.0009]],

         [[ 0.0246]],

         [[ 0.0014]],

         [[-0.0068]],

         [[-0.0106]],

         [[ 0.0090]],

         [[-0.0216]],

         [[ 0.0151]],

         [[ 0.0276]],

         [[ 0.0042]],

         [[ 0.0159]],

         [[ 0.0086]],

    

In [5]:
print("Running mean of layer2.0.bn1:", model.layer2[0].bn1.running_mean)
print("Running variance of layer2.0.bn1:", model.layer2[0].bn1.running_var)

Running mean of layer2.0.bn1: tensor([ 0.1502,  0.3009, -0.1475, -0.1210, -0.5701, -0.7525,  0.0232, -0.1191,
        -0.5203, -0.0344,  0.1527, -0.8009, -0.2133, -0.1956, -0.4503, -0.2632,
         0.0839, -1.3614,  0.3520,  0.0435, -0.5124, -0.4489,  0.3674, -0.7865,
        -0.0061, -0.5502, -0.2629, -0.0697, -0.3892,  0.8596, -0.0261,  0.0194,
        -1.4822,  0.2077,  0.0741, -0.5370,  0.6348,  0.0066, -0.6156, -0.6373,
        -0.2649,  0.3021, -0.6140, -0.8625, -1.1688, -0.2691, -0.7569, -0.7104,
        -0.5601, -0.3803, -0.6424, -0.5653, -0.3943, -0.8532, -0.8817, -0.5444,
        -0.2364, -0.2572, -0.0131, -1.1256,  0.2372, -0.2265, -0.1682, -0.7450,
        -0.8640,  0.2118,  0.1918,  0.5058,  0.0755, -0.6975, -0.7518,  0.5799,
        -0.2933, -0.0071, -0.6256, -0.2616, -0.6733, -1.1375,  0.1193, -0.4987,
        -0.6461, -0.0576,  0.0361,  0.0026, -1.1884,  0.2901, -0.7978, -0.2888,
         0.7106, -0.6718, -0.3914,  0.3720, -0.4927, -0.5238, -0.0162, -0.5074,
        -0

In [59]:
model_vdp.bn1(mu, sigma)

(tensor([[[[-1.2531e+00, -1.6878e+00, -1.6785e+00,  ..., -1.6785e+00,
            -1.6785e+00, -1.4025e+00],
           [ 1.2169e-03,  4.0079e-02,  9.3037e-02,  ...,  9.3037e-02,
             9.3037e-02,  8.0153e-02],
           [ 2.2031e-01,  2.2556e-01,  2.6450e-01,  ...,  2.6450e-01,
             2.6450e-01,  2.5581e-01],
           ...,
           [ 2.2031e-01,  2.2556e-01,  2.6450e-01,  ...,  2.6450e-01,
             2.6450e-01,  2.5581e-01],
           [ 2.2031e-01,  2.2556e-01,  2.6450e-01,  ...,  2.6450e-01,
             2.6450e-01,  2.5581e-01],
           [-1.2262e+00, -1.3215e+00, -1.3276e+00,  ..., -1.3276e+00,
            -1.3276e+00, -5.8236e-01]],
 
          [[-1.2320e+00, -2.2740e+00, -2.3792e+00,  ..., -2.3792e+00,
            -2.3792e+00, -1.7956e+00],
           [ 1.0288e+00,  3.0748e-01,  2.1256e-01,  ...,  2.1256e-01,
             2.1256e-01, -3.8675e-01],
           [ 1.0783e+00,  3.8405e-01,  2.7680e-01,  ...,  2.7680e-01,
             2.7680e-01, -2.7983e-01],


In [45]:
nn.Sequential(*list(model.children())[:2])(input_tensor)

tensor([[[[-3.3038e-01, -5.0254e-01, -4.9885e-01,  ..., -4.9885e-01,
           -4.9885e-01, -3.8956e-01],
          [ 1.6643e-01,  1.8182e-01,  2.0279e-01,  ...,  2.0279e-01,
            2.0279e-01,  1.9769e-01],
          [ 2.5320e-01,  2.5528e-01,  2.7070e-01,  ...,  2.7070e-01,
            2.7070e-01,  2.6726e-01],
          ...,
          [ 2.5320e-01,  2.5528e-01,  2.7070e-01,  ...,  2.7070e-01,
            2.7070e-01,  2.6726e-01],
          [ 2.5320e-01,  2.5528e-01,  2.7070e-01,  ...,  2.7070e-01,
            2.7070e-01,  2.6726e-01],
          [-3.1971e-01, -3.5746e-01, -3.5987e-01,  ..., -3.5987e-01,
           -3.5987e-01, -6.4708e-02]],

         [[-1.7555e-01, -4.1293e-01, -4.3690e-01,  ..., -4.3690e-01,
           -4.3690e-01, -3.0394e-01],
          [ 3.3949e-01,  1.7517e-01,  1.5354e-01,  ...,  1.5354e-01,
            1.5354e-01,  1.7014e-02],
          [ 3.5077e-01,  1.9261e-01,  1.6818e-01,  ...,  1.6818e-01,
            1.6818e-01,  4.1372e-02],
          ...,
     

In [25]:
activation

tensor([[[[0.3385, 0.3222, 0.3314,  ..., 0.3392, 0.3763, 0.3497],
          [0.3115, 0.2958, 0.2967,  ..., 0.3099, 0.3418, 0.3589],
          [0.2467, 0.2508, 0.2497,  ..., 0.2610, 0.3104, 0.3534],
          ...,
          [0.2824, 0.3001, 0.3061,  ..., 0.3156, 0.3444, 0.3632],
          [0.2589, 0.2481, 0.2439,  ..., 0.2617, 0.2752, 0.3072],
          [0.3289, 0.3709, 0.3814,  ..., 0.3877, 0.3804, 0.4082]],

         [[0.4022, 0.2867, 0.2818,  ..., 0.2789, 0.2641, 0.1279],
          [0.5331, 0.4777, 0.4672,  ..., 0.4715, 0.4399, 0.2703],
          [0.5153, 0.4552, 0.4827,  ..., 0.4723, 0.4554, 0.3189],
          ...,
          [0.5136, 0.4718, 0.4981,  ..., 0.4843, 0.4629, 0.3071],
          [0.4433, 0.4000, 0.4014,  ..., 0.3990, 0.4374, 0.2945],
          [0.3255, 0.2979, 0.3340,  ..., 0.3360, 0.3601, 0.3310]],

         [[0.0201, 0.1921, 0.2211,  ..., 0.2104, 0.1090, 0.0000],
          [0.0868, 0.0463, 0.0576,  ..., 0.0430, 0.0104, 0.0000],
          [0.0250, 0.0000, 0.0000,  ..., 0

In [26]:
mu11 

tensor([[[[1.3636e-01, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
           8.2344e-03, 2.2213e-01],
          [6.8472e-01, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
           4.2963e-01, 9.8957e-01],
          [5.6716e-01, 5.4671e-02, 0.0000e+00,  ..., 3.1762e-01,
           8.2276e-01, 1.2599e+00],
          ...,
          [5.6566e-01, 0.0000e+00, 1.7207e-02,  ..., 6.6012e-01,
           1.1583e+00, 1.1296e+00],
          [9.9203e-01, 4.1748e-01, 2.0920e-01,  ..., 7.1134e-01,
           1.0625e+00, 1.1281e+00],
          [5.1318e-01, 7.6228e-02, 5.2438e-02,  ..., 4.6288e-01,
           4.8118e-01, 4.0206e-01]],

         [[1.0969e-02, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [3.1637e-01, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          ...,
          [1.0986e+00, 7.1149e-01, 8.3401e-01,  ..., 5.4991

In [15]:
activations.keys()


dict_keys(['BasicBlock'])

In [15]:
import torch
import torch.nn as nn
import torchvision.models as models

# Load the ResNet18 model
model = models.resnet18(pretrained=True)
model.eval()
# Modify the forward pass to return the penultimate features (before the fully connected layer)
class ResNet18Penultimate(nn.Module):
    def __init__(self, model):
        super(ResNet18Penultimate, self).__init__()
        # Copy the layers up to the average pooling
        self.features = nn.Sequential(*list(model.children())[:-1])  # Exclude the final FC layer
        self.avgpool = model.avgpool  # Retain avgpool for global average pooling

    def forward(self, x):
        # Forward through the feature extractor
        x = self.features(x)
        x = self.avgpool(x)
        # Flatten the output before returning it (penultimate features)
        x = torch.flatten(x, 1)
        return x

# Instantiate the modified model
model_penultimate = ResNet18Penultimate(model)

# Example input tensor (e.g., image with 3 channels, 224x224 size)
input_tensor = torch.randn(1, 3, 224, 224)  # [batch_size, channels, height, width]

# Get the penultimate features
penultimate_features = model_penultimate(input_tensor)

print(f"Penultimate features shape: {penultimate_features.shape}")


Penultimate features shape: torch.Size([1, 512])


/home/jw7630/repos/vdp_modelzoo/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/jw7630/repos/vdp_modelzoo/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [16]:
penultimate_features

tensor([[1.4846e-01, 0.0000e+00, 4.8811e-02, 3.0632e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 1.3377e+00, 6.9949e-01, 0.0000e+00, 0.0000e+00, 1.2545e-02,
         0.0000e+00, 1.6977e-01, 3.8637e-01, 1.5089e-02, 2.7903e-02, 2.5796e-01,
         3.5927e-02, 8.8373e-03, 1.6637e-03, 0.0000e+00, 2.4792e-01, 1.2493e-02,
         4.1201e-02, 4.9875e-02, 5.0348e-01, 8.2864e-02, 1.5347e-01, 1.2203e-01,
         1.3364e-03, 4.0046e-01, 6.0672e-02, 1.4463e-01, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 4.1427e-01, 2.6642e-01, 5.9884e-02, 0.0000e+00, 0.0000e+00,
         1.8986e-03, 1.4909e-02, 5.8295e-02, 2.1578e-01, 4.9068e-01, 1.3326e+00,
         0.0000e+00, 3.2981e-01, 1.9137e-01, 2.2671e-01, 3.3749e-01, 8.0512e-02,
         1.6822e+00, 5.6029e-02, 6.1225e-01, 3.1910e-02, 1.1947e-01, 0.0000e+00,
         0.0000e+00, 3.6715e-02, 8.7756e-02, 2.9154e-01, 4.5492e-01, 7.0359e-01,
         5.8474e-01, 0.0000e+00, 3.0051e-03, 0.0000e+00, 4.2696e-02, 0.0000e+00,
         7.2967e-01, 5.1877e

In [ ]:
[0.7444, 0.6803, 0.8124,  ..., 0.9751, 0.6858, 0.9703],

In [17]:
model_vdp.load_weights(model)
model_vdp.eval()

Parameters assigned successfully.


VDPResNet18(
  (conv1): VDPFirstConv()
  (bn1): VDP_BatchNorm2D()
  (relu): VDP_ReLU()
  (maxpool): VDPMaxPooling()
  (layer11): VDPResnetBasicBlock(
    (conv1): VDPIntermediateConv()
    (bn1): VDP_BatchNorm2D()
    (relu): VDP_ReLU()
    (conv2): VDPIntermediateConv()
    (bn2): VDP_BatchNorm2D()
  )
  (layer12): VDPResnetBasicBlock(
    (conv1): VDPIntermediateConv()
    (bn1): VDP_BatchNorm2D()
    (relu): VDP_ReLU()
    (conv2): VDPIntermediateConv()
    (bn2): VDP_BatchNorm2D()
  )
  (layer21): VDPResnetBasicBlock(
    (conv1): VDPIntermediateConv()
    (bn1): VDP_BatchNorm2D()
    (relu): VDP_ReLU()
    (conv2): VDPIntermediateConv()
    (bn2): VDP_BatchNorm2D()
    (shortcut_conv): VDPIntermediateConv()
    (shortcut_batchnorm): VDP_BatchNorm2D()
  )
  (layer22): VDPResnetBasicBlock(
    (conv1): VDPIntermediateConv()
    (bn1): VDP_BatchNorm2D()
    (relu): VDP_ReLU()
    (conv2): VDPIntermediateConv()
    (bn2): VDP_BatchNorm2D()
  )
  (layer31): VDPResnetBasicBlock(
    (co

In [18]:
model_vdp(input_tensor)

KeyboardInterrupt: 

In [7]:
m2(torch.ones(3,3,244,244))

tensor([[-0.3023,  0.1808, -0.2829,  ..., -0.8013,  0.0009,  0.6212],
        [-0.3023,  0.1808, -0.2829,  ..., -0.8013,  0.0009,  0.6212],
        [-0.3023,  0.1808, -0.2829,  ..., -0.8013,  0.0009,  0.6212]],
       grad_fn=<AddmmBackward0>)

In [6]:
[(n, p.shape) for (n, p) in model_vdp.named_parameters() if "sigma" not in n]

[('conv1.w_mu', torch.Size([64, 3, 7, 7])),
 ('bn1.weight', torch.Size([64])),
 ('bn1.bias', torch.Size([64])),
 ('layer11.conv1.w_mu', torch.Size([64, 64, 3, 3])),
 ('layer11.bn1.weight', torch.Size([64])),
 ('layer11.bn1.bias', torch.Size([64])),
 ('layer11.conv2.w_mu', torch.Size([64, 64, 3, 3])),
 ('layer11.bn2.weight', torch.Size([64])),
 ('layer11.bn2.bias', torch.Size([64])),
 ('layer12.conv1.w_mu', torch.Size([64, 64, 3, 3])),
 ('layer12.bn1.weight', torch.Size([64])),
 ('layer12.bn1.bias', torch.Size([64])),
 ('layer12.conv2.w_mu', torch.Size([64, 64, 3, 3])),
 ('layer12.bn2.weight', torch.Size([64])),
 ('layer12.bn2.bias', torch.Size([64])),
 ('layer21.conv1.w_mu', torch.Size([128, 64, 3, 3])),
 ('layer21.bn1.weight', torch.Size([128])),
 ('layer21.bn1.bias', torch.Size([128])),
 ('layer21.conv2.w_mu', torch.Size([128, 128, 3, 3])),
 ('layer21.bn2.weight', torch.Size([128])),
 ('layer21.bn2.bias', torch.Size([128])),
 ('layer21.shortcut_conv.w_mu', torch.Size([128, 64, 1, 1])

In [10]:
[(n, p.shape) for (n, p) in m2.named_parameters()]

[('conv1.weight', torch.Size([64, 3, 7, 7])),
 ('bn1.weight', torch.Size([64])),
 ('bn1.bias', torch.Size([64])),
 ('layer1.0.conv1.weight', torch.Size([64, 64, 3, 3])),
 ('layer1.0.bn1.weight', torch.Size([64])),
 ('layer1.0.bn1.bias', torch.Size([64])),
 ('layer1.0.conv2.weight', torch.Size([64, 64, 3, 3])),
 ('layer1.0.bn2.weight', torch.Size([64])),
 ('layer1.0.bn2.bias', torch.Size([64])),
 ('layer1.1.conv1.weight', torch.Size([64, 64, 3, 3])),
 ('layer1.1.bn1.weight', torch.Size([64])),
 ('layer1.1.bn1.bias', torch.Size([64])),
 ('layer1.1.conv2.weight', torch.Size([64, 64, 3, 3])),
 ('layer1.1.bn2.weight', torch.Size([64])),
 ('layer1.1.bn2.bias', torch.Size([64])),
 ('layer2.0.conv1.weight', torch.Size([128, 64, 3, 3])),
 ('layer2.0.bn1.weight', torch.Size([128])),
 ('layer2.0.bn1.bias', torch.Size([128])),
 ('layer2.0.conv2.weight', torch.Size([128, 128, 3, 3])),
 ('layer2.0.bn2.weight', torch.Size([128])),
 ('layer2.0.bn2.bias', torch.Size([128])),
 ('layer2.0.downsample.0.we

In [4]:
[n for (n, p) in model_vdp.named_parameters() if "sigma" not in n]

['conv1.w_mu',
 'layer11.conv1.w_mu',
 'layer11.conv1.b_mu',
 'layer11.conv2.w_mu',
 'layer11.conv2.b_mu',
 'layer12.conv1.w_mu',
 'layer12.conv1.b_mu',
 'layer12.conv2.w_mu',
 'layer12.conv2.b_mu',
 'layer21.conv1.w_mu',
 'layer21.conv1.b_mu',
 'layer21.conv2.w_mu',
 'layer21.conv2.b_mu',
 'layer21.shortcut_conv.w_mu',
 'layer21.shortcut_conv.b_mu',
 'layer22.conv1.w_mu',
 'layer22.conv1.b_mu',
 'layer22.conv2.w_mu',
 'layer22.conv2.b_mu',
 'layer22.shortcut_conv.w_mu',
 'layer22.shortcut_conv.b_mu',
 'layer31.conv1.w_mu',
 'layer31.conv1.b_mu',
 'layer31.conv2.w_mu',
 'layer31.conv2.b_mu',
 'layer31.shortcut_conv.w_mu',
 'layer31.shortcut_conv.b_mu',
 'layer32.conv1.w_mu',
 'layer32.conv1.b_mu',
 'layer32.conv2.w_mu',
 'layer32.conv2.b_mu',
 'layer32.shortcut_conv.w_mu',
 'layer32.shortcut_conv.b_mu',
 'layer41.conv1.w_mu',
 'layer41.conv1.b_mu',
 'layer41.conv2.w_mu',
 'layer41.conv2.b_mu',
 'layer41.shortcut_conv.w_mu',
 'layer41.shortcut_conv.b_mu',
 'layer42.conv1.w_mu',
 'layer4

In [9]:
from layers import VDP_BatchNorm2D
bn = VDP_BatchNorm2D()
bn(torch.ones(5,3, 128,128), torch.ones(5,3, 128,128))

(tensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.]],
 
          [[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.]],
 
          [[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.]]],
 
 
         [[[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
           [0., 0., 0

In [11]:
list(bn.named_parameters())

[]

In [4]:
[torch.numel(p) for p in model_vdp.parameters()]

[9408,
 64,
 64,
 64,
 36864,
 64,
 64,
 64,
 36864,
 64,
 64,
 64,
 36864,
 64,
 64,
 64,
 36864,
 64,
 64,
 64,
 73728,
 128,
 128,
 128,
 147456,
 128,
 128,
 128,
 8192,
 128,
 128,
 128,
 147456,
 128,
 128,
 128,
 147456,
 128,
 128,
 128,
 16384,
 128,
 128,
 128,
 294912,
 256,
 256,
 256,
 589824,
 256,
 256,
 256,
 32768,
 256,
 256,
 256,
 589824,
 256,
 256,
 256,
 589824,
 256,
 256,
 256,
 65536,
 256,
 256,
 256,
 1179648,
 512,
 512,
 512,
 2359296,
 512,
 512,
 512,
 131072,
 512,
 512,
 512,
 2359296,
 512,
 512,
 512,
 2359296,
 512,
 512,
 512,
 262144,
 512,
 512,
 512,
 512000,
 1000]

In [10]:
[torch.numel(p) for p in m2.parameters()]

[9408,
 64,
 64,
 36864,
 64,
 64,
 36864,
 64,
 64,
 36864,
 64,
 64,
 36864,
 64,
 64,
 73728,
 128,
 128,
 147456,
 128,
 128,
 8192,
 128,
 128,
 147456,
 128,
 128,
 147456,
 128,
 128,
 294912,
 256,
 256,
 589824,
 256,
 256,
 32768,
 256,
 256,
 589824,
 256,
 256,
 589824,
 256,
 256,
 1179648,
 512,
 512,
 2359296,
 512,
 512,
 131072,
 512,
 512,
 2359296,
 512,
 512,
 2359296,
 512,
 512,
 512000,
 1000]

In [ ]:
import torch.optim as optim
import torch.nn as nn
from utils import mc_nll

EPOCHS = 50
kl_factor = 0.01
model_vdp.train()
optimizer = optim.Adam(model_vdp.parameters(), lr=0.01)
model_vdp = model_vdp.to(device)
for epoch in range(EPOCHS):
    running_loss, running_kl, accuracy = 0.0, 0.0, 0.0
    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        mu, sigma, kl = model_vdp(images)
        loss1 = mc_nll(labels, mu, sigma)
        loss = loss1 + kl_factor * kl
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        running_kl += kl.item()
        accuracy += (mu.argmax(dim=1) == labels).float().sum()
    running_loss /= len(train_loader)
    running_kl /= len(train_loader)
    accuracy /= len(train_loader.dataset)

    if epoch % 10 == 0:
        test_loss, test_kl, test_accuracy = 0.0, 0.0, 00
        for test_images, test_labels in test_loader:
            test_images = test_images.to(device)
            test_labels = test_labels.to(device)
            test_outputs, test_sigmas, kl = model_vdp(test_images)
            test_loss += mc_nll(test_outputs, test_labels).item()
            test_accuracy += (test_outputs.argmax(dim=1) == test_labels).float().sum()
            test_kl += kl.item()
        test_loss /= len(test_loader)
        test_kl /= len(test_loader)
        test_accuracy /= len(test_loader.dataset)
        print(f'Epoch [{epoch+1}/{EPOCHS}], Test Loss: {test_loss:.4f}, kl {test_kl:.4f}Test Accuracy: {test_accuracy:.4f}')
        print(f'Epoch [{epoch+1}/{EPOCHS}], Loss: {running_loss:.4f}, kl {running_kl:.4f}Accuracy: {accuracy:.4f}')

torch.save(model_vdp.state_dict(), 'resnet_vdp.pth')

In [8]:
from torchvision.models import resnet18
m2 = resnet18(num_classes=1000)


In [14]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import pandas as pd
# Example model (use your custom model here)
model = resnet18(pretrained=False)

# Get the parameters and their names
params = list(model.named_parameters())

# Create lists to store the names of the layers and the number of parameters
layer_names = []
param_counts = []

# Loop through the model's parameters and get the names and number of elements
for name, param in params:
    num_params = torch.numel(param)  # Get the total number of parameters in this layer
    layer_names.append(name)
    param_counts.append(num_params)

# Create a DataFrame for better visualization
param_df = pd.DataFrame({
    'Layer Name': layer_names,
    'Number of Parameters': param_counts
})

param_df.head(62)


/home/jw7630/repos/premiumcnn/venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/jw7630/repos/premiumcnn/venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


,Layer Name,Number of Parameters
0,conv1.weight,9408
1,bn1.weight,64
2,bn1.bias,64
3,layer1.0.conv1.weight,36864
4,layer1.0.bn1.weight,64
...,...,...
57,layer4.1.conv2.weight,2359296
58,layer4.1.bn2.weight,512
59,layer4.1.bn2.bias,512
60,fc.weight,512000


In [15]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import pandas as pd
# Example model (use your custom model here)
model = m

# Get the parameters and their names
params = list(model.named_parameters())

# Create lists to store the names of the layers and the number of parameters
layer_names = []
param_counts = []

# Loop through the model's parameters and get the names and number of elements
for name, param in params:
    num_params = torch.numel(param)  # Get the total number of parameters in this layer
    layer_names.append(name)
    param_counts.append(num_params)

# Create a DataFrame for better visualization
param_df = pd.DataFrame({
    'Layer Name': layer_names,
    'Number of Parameters': param_counts
})

param_df.head(62)


,Layer Name,Number of Parameters
0,conv1.w_mu,9408
1,conv1.w_sigma,64
2,layer11.conv1.w_mu,36864
3,layer11.conv1.w_sigma,64
4,layer11.conv2.w_mu,36864
5,layer11.conv2.w_sigma,64
6,layer12.conv1.w_mu,36864
7,layer12.conv1.w_sigma,64
8,layer12.conv2.w_mu,36864
9,layer12.conv2.w_sigma,64


In [11]:
[torch.numel(p) for p in list(m.parameters())]

[9408,
 64,
 36864,
 64,
 36864,
 64,
 36864,
 64,
 36864,
 64,
 73728,
 128,
 147456,
 128,
 8192,
 128,
 147456,
 128,
 147456,
 128,
 16384,
 128,
 294912,
 256,
 589824,
 256,
 32768,
 256,
 589824,
 256,
 589824,
 256,
 65536,
 256,
 1179648,
 512,
 2359296,
 512,
 131072,
 512,
 2359296,
 512,
 2359296,
 512,
 262144,
 512,
 512000,
 1000]

In [4]:
import torchvision.models as models

vgg = models.vgg11(pretrained=True)
list(vgg.named_parameters()) 

/usr/lib/python3/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/lib/python3/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG11_Weights.IMAGENET1K_V1`. You can also use `weights=VGG11_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


[('features.0.weight',
  Parameter containing:
  tensor([[[[ 0.2882,  0.0358, -0.3850],
            [ 0.1795,  0.3668, -0.5012],
            [-0.0974,  0.3648, -0.2296]],
  
           [[ 0.4015, -0.0461, -0.6842],
            [ 0.4442,  0.4478, -0.7949],
            [ 0.1129,  0.4917, -0.3705]],
  
           [[ 0.2162, -0.0290, -0.3949],
            [ 0.1490,  0.2967, -0.4294],
            [-0.0095,  0.3479, -0.1558]]],
  
  
          [[[-0.3528, -0.2549,  0.6734],
            [-0.6027, -0.3453,  0.8054],
            [-0.4197, -0.1347,  0.6622]],
  
           [[-0.5740, -0.3998,  0.7708],
            [-0.8758, -0.3668,  1.1098],
            [-0.5186, -0.0801,  0.9228]],
  
           [[-0.0280, -0.2954,  0.2159],
            [-0.1868, -0.2904,  0.3808],
            [-0.0926, -0.0766,  0.3727]]],
  
  
          [[[ 0.0650, -0.2592, -0.2389],
            [ 0.3209,  0.2152, -0.2047],
            [-0.0443,  0.2227, -0.0452]],
  
           [[ 0.0290, -0.4797, -0.5490],
            [ 0

In [7]:
sum([torch.numel(p) for p in list(m.parameters())[::2]]), sum([torch.numel(p) for p in m2.parameters()])

(12022976, 11689512)

In [2]:
m(torch.randn(1, 3, 224, 224))

(tensor([[-4.9270e-01,  6.4582e-02,  1.0913e-01,  8.9851e-01, -4.2550e-01,
          -1.4902e-02,  4.6031e-01,  8.8639e-01,  1.9970e-01,  1.1081e-01,
          -2.5896e-01,  3.3603e-01, -1.2885e-02, -3.0062e-02,  2.8963e-02,
           2.4176e-01, -4.3537e-01, -1.8880e-01, -3.2406e-01,  9.8007e-01,
           5.5040e-01,  1.7514e-01,  3.5710e-01, -3.7835e-02, -3.6993e-01,
          -1.0509e-01, -7.9113e-01,  1.2079e-01,  8.2023e-02, -9.5774e-02,
           4.7705e-01,  6.0341e-01, -7.8594e-01, -6.9560e-01, -8.1689e-02,
          -4.4115e-01, -1.5033e-01, -7.3954e-01,  1.1759e+00, -2.1287e-01,
          -4.3943e-02,  3.8909e-01, -1.3165e-01,  9.1209e-01,  6.4426e-01,
           1.7369e-01,  4.4050e-01, -7.5946e-02,  1.9973e-01,  2.4482e-01,
          -9.6918e-02, -1.9461e-01,  4.8121e-01, -6.9234e-01,  5.3580e-01,
          -3.7204e-01, -5.3627e-01,  4.4752e-01,  6.1506e-01,  6.3025e-01,
          -8.9139e-01,  5.5846e-01, -7.4304e-01,  2.7076e-01,  7.5750e-01,
           1.1297e+00, -3